# Лабораторная работа 8

Выбранный датасет: Технические задания и статьи по ИТ (кластеризация, классификация).

Выбранный метод машинного обучения: классификация.

Задача анализа текстов: разработка модели, которая сможет автоматически определять категорию, к которой относится текст (в данном случае, ТЗ или статья).

Импорт библиотеки и инициализация модуля для анализа текста:

In [7]:
import spacy

sp = spacy.load("ru_core_news_lg")

Загрузка текстов из файлов с расширением .docx в датафрейм:

In [8]:
import pandas as pd
from docx import Document
import os

def read_docx(file_path):
    doc = Document(file_path)
    full_text = []
    for paragraph in doc.paragraphs:
        full_text.append(paragraph.text)
    return "\n".join(full_text)

def load_docs(dataset_path):
    df = pd.DataFrame(columns=["doc", "text"])
    for file_path in os.listdir(dataset_path):
        if file_path.startswith("~$"):
            continue
        text = read_docx(dataset_path + file_path)
        df.loc[len(df.index)] = [file_path, text]
    return df

df = load_docs("./data/text/")
df["type"] = df.apply(
    lambda row: 0 if str(row["doc"]).startswith("tz_") else 1, axis=1
)
df.sort_values(by=["doc"], inplace=True)

print(df.iloc[15:25])

                                                  doc  \
15                                         tz_16.docx   
16                                         tz_17.docx   
17                                         tz_18.docx   
18                                         tz_19.docx   
19                                         tz_20.docx   
20               Архитектура, управляемая модель.docx   
21                  Введение в проектирование ИС.docx   
22                      Встроенные операторы SQL.docx   
23  Методологии разработки программного обеспечени...   
24  Методологии разработки программного обеспечени...   

                                                 text  type  
15  2.2\tТехническое задание\n2.2.1\tОбщие сведени...     0  
16  2.2 Техническое задание.\n2.2.1 Общие сведения...     0  
17  2.2. Техническое задание\nОбщие сведения:\nПол...     0  
18  2.2. Техническое задание\n2.2.1. Наименование ...     0  
19  2.2. Техническое задание\n2.2.1. Общие сведени...     0  


Предобработка текста.

Трансформация:

In [9]:
import re
import emoji
from num2words import num2words

# Функция для преобразования эмоджи в слова
def emojis_words(text):
    
    # Модуль emoji: преобразование эмоджи в их словесные описания
    text = emoji.demojize(text, delimiters=(" ", " "))
    
    # Редактирование текста путём замены ":" и" _", а так же - путём добавления пробела между отдельными словами
    text = text.replace(":", "").replace("_", " ")
    
    return text

def transform_text(text):
    # Удаление из текста всех HTML-тегов
    text = re.sub(r'<[^<]+?>', '', text)
    
    # Удаление из текста всех URL и ссылок
    text = re.sub(r'http\S+', '', text)

    # Преобразование эмоджи в текст
    text = emojis_words(text)

    # Приведение к нижнему регистру
    text = text.lower()

    # Удаление лишних пробелов
    text = re.sub(r'\s+', ' ', text)  
    
    # Преобразование "ё" в "е"
    text = text.replace("ё", "е")

    # Удаление всех специальных символов
    text = re.sub(r'[^a-zA-Zа-яА-Я0-9\s]', '', text)

    # Преобразование чисел в слова
    words: list[str] = text.split()
    words = [num2words(word, lang="ru") if word.isdigit() else word for word in words]
    text = " ".join(words)

    # Удаление из текста всех знаков препинания
    text = re.sub(r'[^\w\s]', '', text)

    return text

df["preprocessed_text"] = df["text"].apply(transform_text)

Для выполнения токенизации, определения частей речи (POS tagging), нормализации (в данном случае применяется лемматизация) и фильтрации данных будем использовать библиотеку spaCy. На этапе фильтрации с целью уменьшения размерности пространства признаков задействуем словарь стоп-слов, а также исключим все слова, длина которых превышает 20 символов.

In [10]:

from nltk.corpus import stopwords


stop_words = set(stopwords.words('russian'))

def preprocess_text(text):
    doc = sp(text)
    
    filtered_tokens = [

        f"{token.lemma_}_{token.pos_}_{token.morph}" # Формирование строки с нужным форматом
        for token in doc

        if token.text not in stop_words and len(token.text) <= 20 # Фильтрация 

    ]
    

    return " ".join(filtered_tokens)


df["preprocessed_text"] = df["preprocessed_text"].apply(preprocess_text)


# Выведем 10 токенов из первого текста

first_text_tokens = df["preprocessed_text"].iloc[0].split()[:10]

print(" ".join(first_text_tokens))

двадцать_NUM_Case=Nom технический_ADJ_Case=Nom|Degree=Pos|Gender=Neut|Number=Sing задание_NOUN_Animacy=Inan|Case=Nom|Gender=Neut|Number=Sing двести_NUM_Case=Nom двадцать_NUM_Case=Nom общий_ADJ_Case=Nom|Degree=Pos|Number=Plur сведение_NOUN_Animacy=Inan|Case=Gen|Gender=Neut|Number=Sing полный_ADJ_Case=Nom|Degree=Pos|Gender=Neut|Number=Sing наименование_NOUN_Animacy=Inan|Case=Nom|Gender=Neut|Number=Sing система_NOUN_Animacy=Inan|Case=Gen|Gender=Fem|Number=Sing


Перейдем к этапу формирования N-грамм:


In [11]:
import nltk
from nltk.util import ngrams
from nltk.tokenize import word_tokenize
nltk.download("punkt_tab")
def generate_ngrams(text: str, n: int = 2) -> list[tuple]:
    tokens: list[str] = word_tokenize(text, language="russian")
    
    n_grams: list[tuple] = list(ngrams(tokens, n))
    return n_grams

# Пример для биграмм (N=2)
df["bigrams"] = df["preprocessed_text"].apply(lambda x: generate_ngrams(x, n=2))

# Пример для триграмм (N=3)
df["trigrams"] = df["preprocessed_text"].apply(lambda x: generate_ngrams(x, n=3))

print(df.iloc[15:25])

[nltk_data] Downloading package punkt_tab to
[nltk_data]     D:\Users\Leo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


                                                  doc  \
15                                         tz_16.docx   
16                                         tz_17.docx   
17                                         tz_18.docx   
18                                         tz_19.docx   
19                                         tz_20.docx   
20               Архитектура, управляемая модель.docx   
21                  Введение в проектирование ИС.docx   
22                      Встроенные операторы SQL.docx   
23  Методологии разработки программного обеспечени...   
24  Методологии разработки программного обеспечени...   

                                                 text  type  \
15  2.2\tТехническое задание\n2.2.1\tОбщие сведени...     0   
16  2.2 Техническое задание.\n2.2.1 Общие сведения...     0   
17  2.2. Техническое задание\nОбщие сведения:\nПол...     0   
18  2.2. Техническое задание\n2.2.1. Наименование ...     0   
19  2.2. Техническое задание\n2.2.1. Общие сведени...    

Теперь применим методы для векторизации текста.



Мешок слов:

In [12]:
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

counts_vectorizer = CountVectorizer()
counts_matrix = sparse.csr_matrix(counts_vectorizer.fit_transform(df["preprocessed_text"]))
counts_df = pd.DataFrame(
    counts_matrix.toarray(),
    columns=counts_vectorizer.get_feature_names_out(),
)

random_columns = np.random.choice(counts_df.columns, size=10, replace=False)

print(counts_df.loc[15:25, random_columns]) 

    тутто_noun_animacy  взаимоотношение_noun_animacy  иннкпп_propn_animacy  \
15                   0                             0                     0   
16                   0                             0                     0   
17                   0                             0                     0   
18                   0                             0                     0   
19                   0                             0                     0   
20                   0                             0                     0   
21                   0                             1                     0   
22                   0                             0                     0   
23                   0                             0                     0   
24                   0                             0                     0   
25                   0                             0                     0   

    gif_propn_foreign  накладывать_verb_aspect  \
15           

Частотный портрет:

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(sublinear_tf=True)
tfidf_matrix = sparse.csr_matrix(tfidf_vectorizer.fit_transform(df["preprocessed_text"]))
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
)

print(tfidf_df.loc[15:25, random_columns]) 

    тутто_noun_animacy  взаимоотношение_noun_animacy  иннкпп_propn_animacy  \
15                 0.0                      0.000000                   0.0   
16                 0.0                      0.000000                   0.0   
17                 0.0                      0.000000                   0.0   
18                 0.0                      0.000000                   0.0   
19                 0.0                      0.000000                   0.0   
20                 0.0                      0.000000                   0.0   
21                 0.0                      0.022338                   0.0   
22                 0.0                      0.000000                   0.0   
23                 0.0                      0.000000                   0.0   
24                 0.0                      0.000000                   0.0   
25                 0.0                      0.000000                   0.0   

    gif_propn_foreign  накладывать_verb_aspect  \
15           

### Обучение модели и проверка ее качества:

In [14]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def train_and_evaluate(X, y, test_size=0.2, cv=5, optimize=False):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=9)

    if optimize:
        param_grid = {
            "n_estimators": [10, 20, 30, 40, 50, 100, 150, 200, 250, 500],
            "max_features": ["sqrt", "log2", 2],
            "max_depth": [2, 3, 4, 5, 6, 7, 8, 9, 10],
            "criterion": ["gini", "entropy", "log_loss"],
            "class_weight": ["balanced", "balanced_subsample"]
        }

        grid_search = GridSearchCV(RandomForestClassifier(random_state=9), param_grid, scoring="f1", cv=cv, n_jobs=-1)
        grid_search.fit(X_train, y_train)
        model = grid_search.best_estimator_
        print(f"Лучшие параметры: {grid_search.best_params_}")
    else:
        model = RandomForestClassifier(n_estimators=100, random_state=9)
        model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}")

    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1')
    f1_cv = scores.mean()
    print(f"Cross-validated F1 Score: {f1_cv:.4f}")

    return model

X_tfidf = tfidf_df
X_counts = counts_df
y = df["type"]

print("TF-IDF Model")
model_tfidf = train_and_evaluate(X_tfidf, y)

print("\nCount Vectorizer Model")
model_counts = train_and_evaluate(X_counts, y)

TF-IDF Model
Accuracy: 0.8889
Precision: 0.7500
Recall: 1.0000
F1 Score: 0.8571
ROC AUC: 0.9167
Cross-validated F1 Score: 1.0000

Count Vectorizer Model
Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
F1 Score: 1.0000
ROC AUC: 1.0000
Cross-validated F1 Score: 0.9333


Как видно, обе модели демонстрируют отличные результаты, причём вторая модель достигает практически идеальных показателей. Однако это может быть связано с небольшим объёмом данных в выборке (всего 41 документ). Вероятно, модель просто запомнила данные, что привело к её переобучению.

Кроме того, в соответствии с заданием, оценим решение, используя альтернативные гиперпараметры модели машинного обучения, которые будут подобраны с помощью метода поиска по сетке.

In [15]:
print("TF-IDF Model (Optimized)")
model_tfidf = train_and_evaluate(X_tfidf, y, optimize=True)

TF-IDF Model (Optimized)
Лучшие параметры: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 2, 'max_features': 'sqrt', 'n_estimators': 20}
Accuracy: 0.7778
Precision: 0.6000
Recall: 1.0000
F1 Score: 0.7500
ROC AUC: 0.8333
Cross-validated F1 Score: 1.0000


Можно сделать вывод, что в данном случае существует возможность настроить гиперпараметры модели таким образом, что её показатели согласно метрикам достигнут практически идеального уровня.